# Train a microWakeWord model on Google Colab (Python 3.12)

This instructional notebook trains and exports a custom microWakeWord model. Start a **fresh Colab Python 3.12 GPU runtime**, then run cells from top to bottom. The default `SMOKE_TEST = True` exercises synthesis through TFLite export quickly; change it to `False` for useful training volumes. Full training mounts Google Drive and saves resumable model, optimizer, and step checkpoints there. Model quality still requires substantially more data and tuning.

For a fresh run, set `RUN_ACTION = "new"`. `RUN_NAME` is derived from a filename-safe form of `TARGET_WORD` plus `MODEL_VERSION`; increment the version for another fresh run of the same target. To continue an interrupted run, keep the original target and version, then set `RUN_ACTION = "resume"`; metadata checks prevent accidentally resuming a different run.

A 60-second resource sampler records RAM, disk, and available NVIDIA GPU measurements. The final cell writes `run_summary.json`; interrupted runs retain the incremental JSONL event and resource logs in the run directory.

The dependency cell installs code from this fork's `fix/colab-python312` branch. Change `REPOSITORY_REF` only when intentionally testing another revision. No runtime restart is normally required because setup occurs before TensorFlow or PyTorch is imported. If Colab reports that a previously imported package cannot be replaced, restart once and resume at the diagnostics/verification cell.

In [ ]:
# User-configurable settings
SMOKE_TEST = True
TARGET_WORD = "khum_puter"  # phonetic spellings may synthesize better
REPOSITORY_URL = "https://github.com/CJMarais/micro-wake-word.git"
REPOSITORY_REF = "fix/colab-python312"

# Piper Sample Generator model selection. Verify alternatives at PIPER_MODEL_CATALOG_URL.
PIPER_MODEL_DIRECTORY = "piper-models"
PIPER_MODEL_FILENAME = "en_US-libritts_r-medium.pt"
PIPER_MODEL_RELEASE_TAG = "v2.0.0"
PIPER_MODEL_CONFIG_REF = "v3.2.0"
PIPER_MODEL_CATALOG_URL = f"https://github.com/rhasspy/piper-sample-generator/releases/tag/{PIPER_MODEL_RELEASE_TAG}"
PIPER_LANGUAGE_TAG = PIPER_MODEL_FILENAME.split("-", 1)[0]
TRAINED_LANGUAGES = [PIPER_LANGUAGE_TAG.split("_", 1)[0].lower()]

# ESPHome micro-wake-word manifest settings.
MODEL_AUTHOR = "CJMarais"
MODEL_VERSION = 2
PROBABILITY_CUTOFF = 0.63
SLIDING_WINDOW_SIZE = 5
FEATURE_STEP_SIZE = 10
TENSOR_ARENA_SIZE = 22860

# Smoke mode proves the pipeline, not model quality.
SYNTHETIC_SAMPLE_COUNT = 24 if SMOKE_TEST else 1000
TRAINING_STEPS = 2 if SMOKE_TEST else 10000
BATCH_SIZE = 8 if SMOKE_TEST else 128
MAX_AUGMENTATION_FILES = 8 if SMOKE_TEST else None
AUDIOSET_SAMPLE_COUNT = 2000  # full mode; increase for more background diversity

# Persistent full-training run selection.
USE_GOOGLE_DRIVE = not SMOKE_TEST
RUN_ACTION = "new"  # "new" or "resume"
SAFE_TARGET_WORD = "".join(character if character.isascii() and (character.isalnum() or character in "._-") else "_" for character in TARGET_WORD)
while "__" in SAFE_TARGET_WORD:
    SAFE_TARGET_WORD = SAFE_TARGET_WORD.replace("__", "_")
SAFE_TARGET_WORD = SAFE_TARGET_WORD.strip("._-")
if not SAFE_TARGET_WORD:
    raise ValueError("TARGET_WORD must contain at least one filename-safe ASCII letter or number")
RUN_NAME = f"{SAFE_TARGET_WORD}_v{MODEL_VERSION}"
CHECKPOINT_STEP_INTERVAL = 100
RESOURCE_SAMPLE_INTERVAL_S = 60
PERFORMANCE_RUN_LABEL = "free-tier-gpu"  # e.g. free-tier-gpu or free-tier-cpu
PERFORMANCE_NOTES = ""  # optional observations not detectable by the notebook

In [ ]:
# Environment diagnostics and centralized imports (keep this output with bug reports)
import importlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import threading
import time
import urllib.request
import wave
import zipfile
from datetime import datetime, timezone
from importlib.metadata import version
from itertools import islice
from pathlib import Path
try:
    from google.colab import drive as setup_drive
except ImportError:
    setup_drive = None

def load_installed_dependencies():
    # Deferred until setup installs the pinned Colab environment.
    global Audio, Augmentation, Clips, RaggedMmap, SpectrogramGeneration
    global audiomentations, colab_drive, colab_files, datasets, display
    global microwakeword, np, save_clip, scipy, sf, tf, torch, torchaudio, tqdm, yaml
    import audiomentations
    import datasets
    import microwakeword
    import numpy as np
    import scipy
    import scipy.io.wavfile
    import soundfile as sf
    import tensorflow as tf
    import torch
    import torchaudio
    import yaml
    from IPython.display import Audio, display
    from mmap_ninja.ragged import RaggedMmap
    from microwakeword.audio.augmentation import Augmentation
    from microwakeword.audio.audio_utils import save_clip
    from microwakeword.audio.clips import Clips
    from microwakeword.audio.spectrograms import SpectrogramGeneration
    from tqdm.auto import tqdm
    try:
        from google.colab import drive as colab_drive, files as colab_files
    except ImportError:
        colab_drive = colab_files = None

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
print("Executable:", sys.executable)
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("This Colab notebook requires Python 3.12.")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"], check=True)
else:
    print("nvidia-smi: unavailable (select a GPU runtime for normal training)")

In [ ]:
# Environment/dependency implementation details
WORKSPACE = Path("/content/micro-wake-word-work") if Path("/content").is_dir() else Path.cwd() / ".notebook-work"
REPO_DIR = WORKSPACE / "micro-wake-word"
WORKSPACE.mkdir(parents=True, exist_ok=True)

if RUN_ACTION not in {"new", "resume"}:
    raise ValueError('RUN_ACTION must be either "new" or "resume"')
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", RUN_NAME):
    raise ValueError("RUN_NAME must contain only letters, numbers, dot, underscore, or hyphen")
if Path(PIPER_MODEL_DIRECTORY).name != PIPER_MODEL_DIRECTORY:
    raise ValueError("PIPER_MODEL_DIRECTORY must be a single directory name")
if not re.fullmatch(r"[a-z]{2,3}_[A-Z]{2}-[A-Za-z0-9_]+-(?:low|medium|high)\.pt", PIPER_MODEL_FILENAME):
    raise ValueError(f"Unexpected Piper model filename format: {PIPER_MODEL_FILENAME!r}")
if not re.fullmatch(r"[a-z]{2,3}", TRAINED_LANGUAGES[0]):
    raise ValueError(f"Could not derive a base language code from {PIPER_MODEL_FILENAME!r}")

if USE_GOOGLE_DRIVE:
    if not Path("/content").is_dir():
        raise RuntimeError("Google Drive persistence is available only in Colab")
    if setup_drive is None:
        raise RuntimeError("google.colab.drive is unavailable in this runtime")
    setup_drive.mount("/content/drive")
    PERSISTENT_RUNS_ROOT = Path("/content/drive/MyDrive/micro-wake-word/training_runs")
else:
    PERSISTENT_RUNS_ROOT = WORKSPACE / "training_runs"
PERSISTENT_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_RUN_DIR = PERSISTENT_RUNS_ROOT / RUN_NAME
PERSISTENT_RUN_DIR.mkdir(parents=True, exist_ok=True)
RUN_METADATA = PERSISTENT_RUN_DIR / "run_metadata.json"
SESSION_MARKER = WORKSPACE / f".initialized_{RUN_NAME}"
EVENT_LOG = PERSISTENT_RUN_DIR / "execution_events.jsonl"
RESOURCE_LOG = PERSISTENT_RUN_DIR / "resource_samples.jsonl"
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
NOTEBOOK_STARTED_AT = time.time()
RESOURCE_MONITOR_STOP = threading.Event()

def append_jsonl(path, payload):
    with Path(path).open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(payload, sort_keys=True) + "\n")

def read_session_jsonl(path):
    records = []
    for line_number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            print(f"Skipping malformed log record {path}:{line_number}")
            continue
        if record.get("session_id") == SESSION_ID:
            records.append(record)
    return records

def record_event(stage, message):
    event = {"session_id": SESSION_ID, "timestamp_utc": datetime.now(timezone.utc).isoformat(), "elapsed_s": round(time.time() - NOTEBOOK_STARTED_AT, 1), "stage": stage, "message": str(message)}
    append_jsonl(EVENT_LOG, event)

def resource_snapshot():
    memory = {}
    meminfo = Path("/proc/meminfo")
    if meminfo.exists():
        values = {line.split(":", 1)[0]: int(line.split()[1]) for line in meminfo.read_text().splitlines() if line.split()[1:2]}
        memory = {"ram_total_mb": round(values.get("MemTotal", 0) / 1024, 1), "ram_available_mb": round(values.get("MemAvailable", 0) / 1024, 1)}
    disk = shutil.disk_usage(WORKSPACE)
    sample = {"session_id": SESSION_ID, "timestamp_utc": datetime.now(timezone.utc).isoformat(), "elapsed_s": round(time.time() - NOTEBOOK_STARTED_AT, 1), **memory, "disk_used_gb": round(disk.used / 2**30, 2), "disk_total_gb": round(disk.total / 2**30, 2)}
    if shutil.which("nvidia-smi"):
        gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"], text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
        if gpu.returncode == 0 and gpu.stdout.strip():
            name, utilization, used, total = [part.strip() for part in gpu.stdout.splitlines()[0].split(",")]
            sample.update({"gpu_name": name, "gpu_utilization_percent": float(utilization), "gpu_memory_used_mb": float(used), "gpu_memory_total_mb": float(total)})
    return sample

def monitor_resources():
    while not RESOURCE_MONITOR_STOP.is_set():
        try:
            append_jsonl(RESOURCE_LOG, resource_snapshot())
        except Exception as error:
            record_event("resource_monitor", f"sampling failed: {error}")
        RESOURCE_MONITOR_STOP.wait(RESOURCE_SAMPLE_INTERVAL_S)

if not SMOKE_TEST:
    if RUN_ACTION == "new":
        if RUN_METADATA.exists() and not SESSION_MARKER.exists():
            raise FileExistsError(
                f"Run {RUN_NAME!r} already exists. Set RUN_ACTION='resume' or increment MODEL_VERSION."
            )
        PERSISTENT_RUN_DIR.mkdir(parents=True, exist_ok=True)
        if not RUN_METADATA.exists():
            RUN_METADATA.write_text(json.dumps({"schema_version": 1, "run_name": RUN_NAME, "target_word": TARGET_WORD, "repository_url": REPOSITORY_URL, "repository_ref": REPOSITORY_REF, "created_utc": datetime.now(timezone.utc).isoformat()}, indent=2), encoding="utf-8")
        metadata = json.loads(RUN_METADATA.read_text(encoding="utf-8"))
        if metadata.get("run_name") != RUN_NAME or metadata.get("target_word") != TARGET_WORD:
            raise ValueError(f"Existing metadata does not match RUN_NAME={RUN_NAME!r} and TARGET_WORD={TARGET_WORD!r}")
        SESSION_MARKER.touch()
        print(f"Starting NEW training run {RUN_NAME!r} for target {TARGET_WORD!r}")
    else:
        if not RUN_METADATA.exists():
            raise FileNotFoundError(f"Cannot resume {RUN_NAME!r}: run_metadata.json was not found")
        metadata = json.loads(RUN_METADATA.read_text(encoding="utf-8"))
        if metadata.get("run_name") != RUN_NAME:
            raise ValueError(f"Run metadata name {metadata.get('run_name')!r} does not match {RUN_NAME!r}")
        if metadata.get("target_word") != TARGET_WORD:
            raise ValueError(f"Run {RUN_NAME!r} belongs to target {metadata.get('target_word')!r}, not {TARGET_WORD!r}")
        if not list((PERSISTENT_RUN_DIR / "model" / "restore").glob("ckpt-*.index")):
            raise FileNotFoundError(f"Cannot resume {RUN_NAME!r}: no TensorFlow checkpoint was found")
        print(f"RESUMING training run {RUN_NAME!r} for target {TARGET_WORD!r}")

record_event("setup", f"run_action={RUN_ACTION}; run_name={RUN_NAME}; target_word={TARGET_WORD}")
RESOURCE_MONITOR_THREAD = threading.Thread(target=monitor_resources, name="resource-monitor", daemon=True)
RESOURCE_MONITOR_THREAD.start()

def run(command, *, cwd=None, capture=False, quiet=False, stream=False):
    if not quiet:
        print("+", " ".join(map(str, command)))
    if stream:
        process = subprocess.Popen(
            [str(part) for part in command], cwd=cwd, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
        )
        assert process.stdout is not None
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
            if any(marker in line for marker in ("Training progress:", "checkpoint saved", "Restored checkpoint", "Validation progress:", "Ambient validation progress:", "Traceback", "ERROR")):
                record_event("subprocess", line.strip())
        return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, process.args)
        return subprocess.CompletedProcess(process.args, return_code)
    if not capture:
        return subprocess.run([str(part) for part in command], cwd=cwd, check=True, text=True)
    result = subprocess.run([str(part) for part in command], cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout and (not quiet or result.returncode != 0):
        print(result.stdout)
    result.check_returncode()
    return result

def download(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    temporary.unlink(missing_ok=True)
    headers = {"User-Agent": "micro-wake-word-colab/1"}
    if "huggingface.co" in url and os.environ.get("HF_TOKEN"):
        headers["Authorization"] = f"Bearer {os.environ['HF_TOKEN']}"
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers=headers)) as response, temporary.open("wb") as output:
            shutil.copyfileobj(response, output)
        if temporary.stat().st_size == 0:
            raise RuntimeError("downloaded file is empty")
        temporary.replace(destination)
    except Exception as error:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Failed to download {url} to {destination}: {error}") from error
    return destination

# Explicit codecs are required for FLAC/MP3 conversion in fresh Colab images.
if Path("/content").is_dir():
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1"])

if not (REPO_DIR / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", REPOSITORY_REF, REPOSITORY_URL, REPO_DIR])
else:
    print("Reusing existing checkout:", REPO_DIR)
repository_commit = run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True).stdout.strip()
if not repository_commit:
    raise RuntimeError(f"Could not identify the checked-out repository commit in {REPO_DIR}")
print("Repository commit:", repository_commit)

requirements = REPO_DIR / "notebooks" / "requirements_colab_py312.txt"
if not requirements.is_file():
    raise FileNotFoundError(f"Missing Colab requirements file: {requirements}")
PIPER_SOURCE = WORKSPACE / "piper-sample-generator"
if not (PIPER_SOURCE / ".git").exists():
    run(["git", "clone", "--depth", "1", "--branch", "v3.2.0", "https://github.com/rhasspy/piper-sample-generator.git", PIPER_SOURCE])
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", requirements])
# Dependencies were resolved above as one coherent set; install this exact checkout.
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", REPO_DIR])
# Piper's source tree also supplies the piper_train package imported by its CLI.
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", PIPER_SOURCE])

# Confirm that the project package is importable. If an earlier interrupted
# setup left it absent, repair only this editable installation.
repo_import_path = str(REPO_DIR.resolve())
if repo_import_path not in sys.path:
    # Editable-install .pth files are processed only when Python starts. Add
    # this exact checkout to the already-running Colab kernel immediately.
    sys.path.insert(0, repo_import_path)
    importlib.invalidate_caches()
if importlib.util.find_spec("microwakeword") is None:
    print("microwakeword is not importable; reinstalling the checked-out project")
    run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", REPO_DIR])
    importlib.invalidate_caches()
load_installed_dependencies()
microwakeword_path = Path(microwakeword.__file__).resolve()
assert microwakeword_path.is_relative_to(REPO_DIR.resolve()), (
    f"Loaded microwakeword from {microwakeword_path}, expected checkout {REPO_DIR}"
)
print("microWakeWord import verified:", microwakeword_path)
os.chdir(WORKSPACE)
print("Working directory:", Path.cwd())
record_event("setup", "dependencies installed and repository import verified")

In [ ]:
# Verify the resolved runtime before doing expensive work.
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__, "GPUs:", tf.config.list_physical_devices("GPU"))
print("PyTorch:", torch.__version__, "torchaudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available(), "CUDA:", torch.version.cuda)
print("datasets:", datasets.__version__, "audiomentations:", version("audiomentations"))
print("Piper Sample Generator:", version("piper-sample-generator"))
assert np.lib.NumpyVersion(np.__version__) >= np.lib.NumpyVersion("2.0.0")
assert tf.__version__.startswith("2.21.")
required_augmentations = ["AddBackgroundNoise", "AddColorNoise", "ApplyImpulseResponse", "BandStopFilter", "Gain", "GainTransition", "Normalize", "PitchShift", "SevenBandParametricEQ", "TanhDistortion"]
missing_augmentations = [name for name in required_augmentations if not hasattr(audiomentations, name)]
assert not missing_augmentations, (
    f"audiomentations {version('audiomentations')} is missing required APIs: {missing_augmentations}"
)
print("microWakeWord augmentation API check passed")
record_event("runtime_verification", f"tensorflow={tf.__version__}; torch={torch.__version__}; cuda={torch.cuda.is_available()}")

## Synthesize and verify the wake word

The first command generates **exactly one** WAV and validates it before any bulk generation. It uses Piper 3.2.0's module entry point from a pinned editable source checkout because the CLI imports its bundled `piper_train` package. Generator-ready language models are listed on the [Piper Sample Generator release page](https://github.com/rhasspy/piper-sample-generator/releases/tag/v2.0.0); set `PIPER_MODEL_FILENAME` to an asset whose matching `.pt.json` file exists in the selected config reference. `TRAINED_LANGUAGES` is derived from the filename's locale prefix.

In [ ]:
PIPER_MODEL_DIR = WORKSPACE / PIPER_MODEL_DIRECTORY
PIPER_MODEL = PIPER_MODEL_DIR / PIPER_MODEL_FILENAME
PIPER_CONFIG = Path(str(PIPER_MODEL) + ".json")
SAMPLE_DIR = WORKSPACE / "generated_samples"
PIPER_MODEL_DIR.mkdir(exist_ok=True)
if not PIPER_MODEL.exists():
    download(f"https://github.com/rhasspy/piper-sample-generator/releases/download/{PIPER_MODEL_RELEASE_TAG}/{PIPER_MODEL_FILENAME}", PIPER_MODEL)
if not PIPER_CONFIG.exists():
    download(f"https://raw.githubusercontent.com/rhasspy/piper-sample-generator/{PIPER_MODEL_CONFIG_REF}/models/{PIPER_MODEL_FILENAME}.json", PIPER_CONFIG)
assert PIPER_MODEL.stat().st_size > 0 and PIPER_CONFIG.stat().st_size > 0, "Piper model/config download is empty"

smoke_dir = WORKSPACE / "piper_one_sample"
if smoke_dir.exists():
    shutil.rmtree(smoke_dir)
run([sys.executable, "-m", "piper_sample_generator", TARGET_WORD, "--model", PIPER_MODEL, "--max-samples", "1", "--batch-size", "1", "--output-dir", smoke_dir], cwd=PIPER_SOURCE, capture=True)
smoke_wavs = list(smoke_dir.glob("*.wav"))
assert len(smoke_wavs) == 1 and smoke_wavs[0].stat().st_size > 44, f"Expected exactly one valid WAV, found {smoke_wavs}"
with wave.open(str(smoke_wavs[0]), "rb") as wav:
    assert wav.getnframes() > 0 and wav.getframerate() > 0
display(Audio(filename=str(smoke_wavs[0]), autoplay=True))
record_event("synthesis_probe", f"validated {smoke_wavs[0].name}")

In [ ]:
# Bulk sample generation (idempotent: regenerate only if the count is short).
SAMPLE_DIR.mkdir(exist_ok=True)
existing = list(SAMPLE_DIR.glob("*.wav"))
if len(existing) < SYNTHETIC_SAMPLE_COUNT:
    shutil.rmtree(SAMPLE_DIR)
    batch = min(8, SYNTHETIC_SAMPLE_COUNT) if SMOKE_TEST else min(100, SYNTHETIC_SAMPLE_COUNT)
    run([sys.executable, "-m", "piper_sample_generator", TARGET_WORD, "--model", PIPER_MODEL, "--max-samples", SYNTHETIC_SAMPLE_COUNT, "--batch-size", batch, "--output-dir", SAMPLE_DIR], cwd=PIPER_SOURCE, capture=True)
generated_wavs = sorted(SAMPLE_DIR.glob("*.wav"))
assert len(generated_wavs) == SYNTHETIC_SAMPLE_COUNT
assert all(path.stat().st_size > 44 for path in generated_wavs)
print(f"Validated {len(generated_wavs)} synthesized WAV files")
record_event("sample_generation", f"validated {len(generated_wavs)} synthesized WAV files")

## Obtain augmentation audio

MIT room impulse responses are always downloaded. Full mode streams the current AudioSet balanced-training Parquet dataset and downloads the FMA MP3 subset, converting both sources to mono 16 kHz PCM WAV. Smoke mode creates small deterministic noise backgrounds so the entire pipeline can be validated without multi-gigabyte downloads; full mode is the dataset/codec integration path for real training. A Hugging Face token is optional; setting `HF_TOKEN` can increase rate limits and download reliability. Review the source dataset licences before distributing or commercially using a model.

In [ ]:
RIR_DIR = WORKSPACE / "mit_rirs"
BACKGROUND_DIR = WORKSPACE / "background_16k"
RIR_DIR.mkdir(exist_ok=True)
BACKGROUND_DIR.mkdir(exist_ok=True)

if not list(RIR_DIR.glob("*.wav")):
    rir_rows = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
    for index, row in enumerate(tqdm(rir_rows, desc="MIT RIR")):
        audio = row["audio"]
        samples = np.asarray(audio["array"], dtype=np.float32)
        assert samples.size and np.isfinite(samples).all()
        name = Path(audio["path"]).stem + ".wav"
        scipy.io.wavfile.write(RIR_DIR / name, 16000, np.clip(samples * 32767, -32768, 32767).astype(np.int16))
        if MAX_AUGMENTATION_FILES and index + 1 >= MAX_AUGMENTATION_FILES:
            break

def convert_audio(inputs, output_dir, limit=None):
    converted = []
    created = 0
    for source in list(inputs)[:limit]:
        destination = output_dir / (source.stem + ".wav")
        if not destination.exists():
            run(["ffmpeg", "-nostdin", "-loglevel", "error", "-i", source, "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le", "-y", destination], capture=True, quiet=True)
            created += 1
        samples, rate = sf.read(destination, dtype="float32")
        assert rate == 16000 and samples.size and np.isfinite(samples).all()
        converted.append(destination)
    print(f"Audio conversion complete: {created} created, {len(converted) - created} reused")
    return converted

if SMOKE_TEST:
    rng = np.random.default_rng(312)
    for index in range(8):
        path = BACKGROUND_DIR / f"smoke_noise_{index}.wav"
        if not path.exists():
            scipy.io.wavfile.write(path, 16000, (rng.normal(0, 0.04, 48000) * 32767).astype(np.int16))
else:
    downloads = WORKSPACE / "downloads"
    downloads.mkdir(exist_ok=True)
    existing_audioset = sorted(BACKGROUND_DIR.glob("audioset_*.wav"))
    if len(existing_audioset) < AUDIOSET_SAMPLE_COUNT:
        audioset_rows = datasets.load_dataset("agkphysics/AudioSet", "balanced", split="train", streaming=True)
        audioset_rows = audioset_rows.cast_column("audio", datasets.Audio(sampling_rate=16000, mono=True))
        selected_rows = islice(audioset_rows, AUDIOSET_SAMPLE_COUNT)
        for index, row in enumerate(tqdm(selected_rows, total=AUDIOSET_SAMPLE_COUNT, desc="AudioSet")):
            destination = BACKGROUND_DIR / f"audioset_{index:05d}.wav"
            if not destination.exists():
                audio = row["audio"]
                samples = np.asarray(audio["array"], dtype=np.float32)
                assert samples.size and np.isfinite(samples).all()
                scipy.io.wavfile.write(destination, 16000, np.clip(samples * 32767, -32768, 32767).astype(np.int16))
    assert len(list(BACKGROUND_DIR.glob("audioset_*.wav"))) >= AUDIOSET_SAMPLE_COUNT
    fma_zip = downloads / "fma_xs.zip"
    download("https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip", fma_zip)
    fma_raw = WORKSPACE / "fma_raw"
    if not fma_raw.exists():
        fma_raw.mkdir(); zipfile.ZipFile(fma_zip).extractall(fma_raw)
    assert convert_audio(fma_raw.rglob("*.mp3"), BACKGROUND_DIR)

assert list(RIR_DIR.glob("*.wav")) and list(BACKGROUND_DIR.glob("*.wav"))
print("RIR files:", len(list(RIR_DIR.glob("*.wav"))), "background files:", len(list(BACKGROUND_DIR.glob("*.wav"))))
record_event("augmentation_audio", f"rir={len(list(RIR_DIR.glob('*.wav')))}; background={len(list(BACKGROUND_DIR.glob('*.wav')))}")

In [ ]:
# Configure and smoke-test the actual microWakeWord audio pipeline.
clips = Clips(str(SAMPLE_DIR), "*.wav", remove_silence=False, random_split_seed=10, split_count=0.1)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={"SevenBandParametricEQ": .1, "TanhDistortion": .1, "PitchShift": .1, "BandStopFilter": .1, "AddColorNoise": .1, "AddBackgroundNoise": .75, "Gain": 1.0, "RIR": .5},
    impulse_paths=[str(RIR_DIR)], background_paths=[str(BACKGROUND_DIR)],
    background_min_snr_db=-5, background_max_snr_db=10, min_jitter_s=.195, max_jitter_s=.205,
)
augmented_clip = augmenter.augment_clip(clips.get_random_clip())
assert augmented_clip.shape == (51200,) and np.isfinite(augmented_clip).all()
AUGMENTED_WAV = WORKSPACE / "augmented_clip.wav"
save_clip(augmented_clip, str(AUGMENTED_WAV))
display(Audio(filename=str(AUGMENTED_WAV), autoplay=True))

feature_probe = SpectrogramGeneration(clips, augmenter, slide_frames=1, step_ms=10).get_random_spectrogram()
assert feature_probe.ndim == 2 and feature_probe.shape[0] > 0 and np.isfinite(feature_probe).all()
print("Feature smoke test shape:", feature_probe.shape)
record_event("audio_pipeline", f"feature_probe_shape={feature_probe.shape}")

In [ ]:
# Generate positive train/validation/test RaggedMmap features.
POSITIVE_FEATURES = WORKSPACE / "generated_augmented_features"
if POSITIVE_FEATURES.exists():
    shutil.rmtree(POSITIVE_FEATURES)
for split, source_split, repetition, slide_frames in [( "training", "train", 1 if SMOKE_TEST else 2, 2 if SMOKE_TEST else 10), ("validation", "validation", 1, 1), ("testing", "test", 1, 1)]:
    out_dir = POSITIVE_FEATURES / split / "wakeword_mmap"
    out_dir.parent.mkdir(parents=True, exist_ok=True)
    spectrograms = SpectrogramGeneration(clips, augmenter, slide_frames=slide_frames, step_ms=10)
    RaggedMmap.from_generator(out_dir=out_dir, sample_generator=spectrograms.spectrogram_generator(split=source_split, repeat=repetition), batch_size=8 if SMOKE_TEST else 100, verbose=True)
    assert out_dir.exists() and any(out_dir.iterdir())
print("Positive feature sets created at", POSITIVE_FEATURES)
record_event("positive_features", str(POSITIVE_FEATURES))

In [ ]:
# Obtain negative features. Smoke mode builds a tiny local set; full mode uses
# the pre-generated microWakeWord feature archives.
NEGATIVE_FEATURES = WORKSPACE / "negative_datasets"
if SMOKE_TEST:
    if NEGATIVE_FEATURES.exists(): shutil.rmtree(NEGATIVE_FEATURES)
    background_clips = Clips(str(BACKGROUND_DIR), "*.wav", random_split_seed=11, split_count=0.1)
    for split, source_split in [("training", "train"), ("validation", "validation"), ("testing", "test")]:
        out_dir = NEGATIVE_FEATURES / "smoke_noise" / split / "noise_mmap"
        out_dir.parent.mkdir(parents=True, exist_ok=True)
        generator = SpectrogramGeneration(background_clips, split_spectrogram_duration_s=3.2, step_ms=10).spectrogram_generator(split=source_split)
        RaggedMmap.from_generator(out_dir=out_dir, sample_generator=generator, batch_size=8, verbose=True)
else:
    NEGATIVE_FEATURES.mkdir(exist_ok=True)
    root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    for name in ["dinner_party.zip", "dinner_party_eval.zip", "no_speech.zip", "speech.zip"]:
        archive = NEGATIVE_FEATURES / name
        download(root + name, archive)
        marker = NEGATIVE_FEATURES / (name + ".extracted")
        if not marker.exists():
            zipfile.ZipFile(archive).extractall(NEGATIVE_FEATURES); marker.touch()
assert any(NEGATIVE_FEATURES.rglob("*mmap")), "No negative feature mmap was created/downloaded"
record_event("negative_features", str(NEGATIVE_FEATURES))

In [ ]:
# Write the training configuration. User tuning belongs here, not in setup.
TRAIN_DIR = (WORKSPACE / "trained_models" / "wakeword") if SMOKE_TEST else PERSISTENT_RUN_DIR / "model"
positive_feature = {"features_dir": str(POSITIVE_FEATURES), "sampling_weight": 2.0, "penalty_weight": 1.0, "truth": True, "truncation_strategy": "truncate_start", "type": "mmap"}
if SMOKE_TEST:
    negative_features = [{"features_dir": str(NEGATIVE_FEATURES / "smoke_noise"), "sampling_weight": 1.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"}]
else:
    negative_features = [
      {"features_dir": str(NEGATIVE_FEATURES / "speech"), "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
      {"features_dir": str(NEGATIVE_FEATURES / "dinner_party"), "sampling_weight": 10.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
      {"features_dir": str(NEGATIVE_FEATURES / "no_speech"), "sampling_weight": 5.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
      {"features_dir": str(NEGATIVE_FEATURES / "dinner_party_eval"), "sampling_weight": 0.0, "penalty_weight": 1.0, "truth": False, "truncation_strategy": "split", "type": "mmap"},
    ]
config = {
 "window_step_ms": 10, "train_dir": str(TRAIN_DIR), "features": [positive_feature, *negative_features],
 "training_steps": [TRAINING_STEPS], "positive_class_weight": [1], "negative_class_weight": [20],
 "learning_rates": [0.001], "batch_size": BATCH_SIZE, "validation_batch_size": BATCH_SIZE, "time_mask_max_size": [0], "time_mask_count": [0],
 "freq_mask_max_size": [0], "freq_mask_count": [0], "eval_step_interval": 1 if SMOKE_TEST else 500,
 "progress_step_interval": 1 if SMOKE_TEST else 100, "validation_progress_batches": 1 if SMOKE_TEST else 100,
 "checkpoint_step_interval": 0 if SMOKE_TEST else CHECKPOINT_STEP_INTERVAL,
 "clip_duration_ms": 1500, "target_minimization": 0.9, "minimization_metric": None, "maximization_metric": "average_viable_recall",
}
TRAINING_CONFIG = WORKSPACE / "training_parameters.yaml"
TRAINING_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print(TRAINING_CONFIG.read_text())
record_event("training_configuration", f"steps={TRAINING_STEPS}; batch_size={BATCH_SIZE}; checkpoint_interval={CHECKPOINT_STEP_INTERVAL}")

In [ ]:
# Train, validate, and convert while streaming subprocess diagnostics.
# Smoke mode is intentionally deterministic: discard only its generated
# training directory so rerunning this cell cannot collide with a partial run.
if SMOKE_TEST and TRAIN_DIR.exists():
    resolved_train_dir = TRAIN_DIR.resolve()
    assert resolved_train_dir.is_relative_to(WORKSPACE.resolve()), (
        f"Refusing to remove training directory outside workspace: {resolved_train_dir}"
    )
    print("Removing previous smoke-test training output:", resolved_train_dir)
    shutil.rmtree(resolved_train_dir)
weights_to_export = "last_weights" if SMOKE_TEST else "best_weights"
training_command = [sys.executable, "-m", "microwakeword.model_train_eval",
 "--training_config", TRAINING_CONFIG, "--train", "1", "--restore_checkpoint", "1" if (not SMOKE_TEST and RUN_ACTION == "resume") else "0",
 "--test_tf_nonstreaming", "0", "--test_tflite_nonstreaming", "0", "--test_tflite_nonstreaming_quantized", "0",
 "--test_tflite_streaming", "0", "--test_tflite_streaming_quantized", "1", "--use_weights", weights_to_export,
 "mixednet", "--pointwise_filters", "64,64,64,64", "--repeat_in_block", "1, 1, 1, 1",
 "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]", "--residual_connection", "0,0,0,0",
 "--first_conv_filters", "32", "--first_conv_kernel_size", "5", "--stride", "3"]
run(training_command, cwd=WORKSPACE, stream=True)
TFLITE_MODEL = TRAIN_DIR / "tflite_stream_state_internal_quant" / "stream_state_internal_quant.tflite"
assert TFLITE_MODEL.exists() and TFLITE_MODEL.stat().st_size > 0, f"Missing/empty export: {TFLITE_MODEL}"
print("Valid TFLite export:", TFLITE_MODEL, TFLITE_MODEL.stat().st_size, "bytes")
record_event("training_complete", f"tflite_bytes={TFLITE_MODEL.stat().st_size}")

In [ ]:
# Persist the verified model using the run name. Fall back to a browser download.
PERSISTENT_MODEL = PERSISTENT_RUN_DIR / f"{RUN_NAME}.tflite"
try:
    PERSISTENT_RUN_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TFLITE_MODEL, PERSISTENT_MODEL)
    assert PERSISTENT_MODEL.stat().st_size == TFLITE_MODEL.stat().st_size
except (OSError, AssertionError) as error:
    PERSISTENT_MODEL.unlink(missing_ok=True)
    print(f"Could not save the model to {PERSISTENT_MODEL}: {error}")
    if colab_files is None:
        print("Fallback download is unavailable; model remains at", TFLITE_MODEL)
    else:
        print("Starting fallback browser download:", TFLITE_MODEL.name)
        colab_files.download(str(TFLITE_MODEL))
else:
    print("Model saved successfully:", PERSISTENT_MODEL)

PERSISTENT_MANIFEST = None
if PERSISTENT_MODEL.exists():
    if not 0.0 < PROBABILITY_CUTOFF < 1.0:
        raise ValueError("PROBABILITY_CUTOFF must be between 0 and 1")
    if min(MODEL_VERSION, SLIDING_WINDOW_SIZE, FEATURE_STEP_SIZE, TENSOR_ARENA_SIZE) < 1:
        raise ValueError("Manifest integer settings must be positive")
    if not TRAINED_LANGUAGES or not all(isinstance(language, str) and language for language in TRAINED_LANGUAGES):
        raise ValueError("TRAINED_LANGUAGES must contain at least one language code")
    model_manifest = {
        "type": "micro",
        "wake_word": TARGET_WORD,
        "author": MODEL_AUTHOR,
        "version": MODEL_VERSION,
        "trained_languages": TRAINED_LANGUAGES,
        "micro": {
            "probability_cutoff": PROBABILITY_CUTOFF,
            "sliding_window_size": SLIDING_WINDOW_SIZE,
            "feature_step_size": FEATURE_STEP_SIZE,
            "tensor_arena_size": TENSOR_ARENA_SIZE,
        },
    }
    PERSISTENT_MANIFEST = PERSISTENT_RUN_DIR / f"{RUN_NAME}.json"
    manifest_temporary = PERSISTENT_MANIFEST.with_suffix(".json.part")
    manifest_temporary.write_text(json.dumps(model_manifest, indent=2) + "\n", encoding="utf-8")
    verified_manifest = json.loads(manifest_temporary.read_text(encoding="utf-8"))
    if verified_manifest != model_manifest:
        manifest_temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Manifest verification failed: {PERSISTENT_MANIFEST}")
    manifest_temporary.replace(PERSISTENT_MANIFEST)
    print("Model manifest saved successfully:", PERSISTENT_MANIFEST)
    record_event("model_manifest", str(PERSISTENT_MANIFEST))
record_event("model_export", str(PERSISTENT_MODEL if PERSISTENT_MODEL.exists() else TFLITE_MODEL))

In [ ]:
# Finalize the low-overhead execution and resource summary.
RESOURCE_MONITOR_STOP.set()
RESOURCE_MONITOR_THREAD.join(timeout=5)
append_jsonl(RESOURCE_LOG, resource_snapshot())
record_event("notebook_complete", "all cells completed")

resource_samples = read_session_jsonl(RESOURCE_LOG)
events = read_session_jsonl(EVENT_LOG)
def sampled_max(key):
    values = [sample[key] for sample in resource_samples if key in sample]
    return max(values) if values else None
resource_summary = {
    "sample_count": len(resource_samples),
    "minimum_ram_available_mb": min((sample["ram_available_mb"] for sample in resource_samples if "ram_available_mb" in sample), default=None),
    "peak_disk_used_gb": sampled_max("disk_used_gb"),
    "peak_gpu_utilization_percent": sampled_max("gpu_utilization_percent"),
    "peak_gpu_memory_used_mb": sampled_max("gpu_memory_used_mb"),
}
git_result = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
summary = {
    "run_name": RUN_NAME,
    "session_id": SESSION_ID,
    "run_action": RUN_ACTION,
    "target_word": TARGET_WORD,
    "performance_run_label": PERFORMANCE_RUN_LABEL,
    "performance_notes": PERFORMANCE_NOTES,
    "smoke_test": SMOKE_TEST,
    "elapsed_s": round(time.time() - NOTEBOOK_STARTED_AT, 1),
    "python": sys.version.replace("\n", " "),
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "repository_url": REPOSITORY_URL,
    "repository_ref": REPOSITORY_REF,
    "repository_commit": git_result.stdout.strip() if git_result.returncode == 0 else None,
    "tensorflow": tf.__version__,
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "training_steps": TRAINING_STEPS,
    "batch_size": BATCH_SIZE,
    "synthetic_sample_count": SYNTHETIC_SAMPLE_COUNT,
    "audioset_sample_count": AUDIOSET_SAMPLE_COUNT,
    "checkpoint_step_interval": CHECKPOINT_STEP_INTERVAL,
    "model_path": str(PERSISTENT_MODEL if PERSISTENT_MODEL.exists() else TFLITE_MODEL),
    "manifest_path": str(PERSISTENT_MANIFEST) if PERSISTENT_MANIFEST else None,
    "resource_sample_interval_s": RESOURCE_SAMPLE_INTERVAL_S,
    "resource_summary": resource_summary,
    "resource_samples": resource_samples,
    "events": events,
}
RUN_SUMMARY = PERSISTENT_RUN_DIR / f"run_summary_{SESSION_ID}.json"
LATEST_RUN_SUMMARY = PERSISTENT_RUN_DIR / "run_summary.json"
summary_text = json.dumps(summary, indent=2)
RUN_SUMMARY.write_text(summary_text, encoding="utf-8")
LATEST_RUN_SUMMARY.write_text(summary_text, encoding="utf-8")
print("Execution summary saved successfully:", RUN_SUMMARY)